# The Ninefold Tide — supporting voice auditions (17 characters)

Generates TWO original synthetic reference-voice options for each of the seventeen characters not covered by the original Naya/Kito/Iri pilot. This reflects the 20-person canon (19 plus Supervisor Ren) from the updated Google Drive director handoff. No real human voice is cloned. All files are private and unapproved until quality evaluation.

Model weights are cached under Kaggle temporary storage, not saved to the output ZIP. Kaggle runs on free quota only; the user's device does not download gigabytes of weights.


In [ ]:
from pathlib import Path
import json,os,sys,subprocess,zipfile,time
from datetime import datetime,timezone
os.environ["HF_HOME"]="/kaggle/temp/ninefold_cast_huggingface_cache"
os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
root=Path("/kaggle/temp/ninefold_cast_source")
out=Path("/kaggle/working/ninefold_cast_20_voice_bank")
out.mkdir(parents=True,exist_ok=True)
status={"project":"The Ninefold Tide", "scope":"17 previously unvoiced recurring characters",
 "approval":"PROVISIONAL_UNAPPROVED", "state":"STARTED",
 "started_utc":datetime.now(timezone.utc).isoformat(), "completed_steps":[]}
def save():
    (out/"batch_status.json").write_text(json.dumps(status,indent=2)+"\n",encoding="utf8")
def step(name,argv,timeout):
    t=time.monotonic()
    print("START",name,flush=True)
    p=subprocess.run(argv,text=True,capture_output=True,timeout=timeout)
    (out/("log_"+name.replace(" ","_")+".txt")).write_text(
        (p.stdout or "")[-18000:]+"\n--- STDERR ---\n"+(p.stderr or "")[-18000:],encoding="utf8")
    print("END",name,"return",p.returncode,"elapsed",round(time.monotonic()-t,1),flush=True)
    if p.returncode:
        raise RuntimeError(name+": "+(p.stderr or p.stdout or "")[-450:])
    status["completed_steps"].append(name)
    save()
save()
try:
    step("Check NVIDIA CUDA",["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv,noheader"],40)
    step("Install Qwen TTS", [sys.executable,"-m","pip","install","-q","qwen-tts","soundfile"],500)
    step("Clone exact series production source",
         ["git","clone","-q","--depth","1","https://github.com/eghosa001/TOOLS.git",str(root)],130)
    worker=root/"video/ninefold_tide/audio/supporting_cast_worker.py"
    step("Generate 34 new supporting actor voice candidates",
         [sys.executable,str(worker),"--out",str(out),"--takes","2","--allow-download"],1600)
    result=json.loads((out/"supporting_voice_manifest.json").read_text(encoding="utf8"))
    status["generated_voice_candidates"]=result["candidate_count"]
    status["state"]="AUDITION_REFERENCES_GENERATED_UNAPPROVED" if result["candidate_count"]==34 else "INCOMPLETE"
except Exception as exc:
    status["state"]="PARTIAL_OR_FAILED"
    status["error"]=str(exc)[:1100]
    print("AUDITION_RUN_BLOCKED",str(exc)[:450],flush=True)
finally:
    status["ended_utc"]=datetime.now(timezone.utc).isoformat()
    save()
    archive=Path("/kaggle/working/ninefold_17_supporting_cast_UNAPPROVED.zip")
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for path in out.rglob("*"):
            if path.is_file() and path.suffix.lower() in (".wav",".json",".txt"):
                z.write(path,arcname=path.relative_to(out))
    print("AUDITION_ARCHIVE",str(archive),archive.stat().st_size,status["state"],flush=True)
    print("All reference voices require later quality assessment. Existing three leads untouched.",flush=True)
